# CytoBridge: developing chicken heart

This notebook prepares or validates the GSE149457 D4/D7/D10/D14 input, loads the `chicken_heart` preset from the installed package, and runs the package workflow when the corresponding switches are enabled. Training and downstream execution are disabled by default.


## 1. Inputs

Set the raw-data and checkpoint paths below. `MODEL_DIR` points to an existing checkpoint; a new fit is written under `TRAIN_OUTPUT_DIR`. `DEVICE_OVERRIDE=None` selects CUDA when it is available and CPU otherwise.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import scanpy as sc
import torch

import CytoBridge as cb
from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)

DATASET_PRESET = "chicken_heart"
RAW_DIR = Path("inputs/GSE149457_RAW")
METADATA_H5AD = Path("inputs/chicken_heart_spatial_merged_with_meta.h5ad")
REFERENCE_ALIGNMENT_H5AD = Path("inputs/heart_aligned_all_timepoints.h5ad")

PREPARED_DIR = Path("tutorial_outputs/chicken_heart/prepared")
PREPARED_H5AD = PREPARED_DIR / "chicken_heart_reference_input.h5ad"
PREPARED_TABLE = PREPARED_DIR / "model_input.csv"
PREPARED_MANIFEST = PREPARED_DIR / "preparation.json"
WORKFLOW_INPUT_H5AD = PREPARED_DIR / "chicken_heart_ot_input.h5ad"
WORKFLOW_INPUT_TABLE = PREPARED_DIR / "chicken_heart_ot_input.csv"
WORKFLOW_INPUT_MANIFEST = PREPARED_DIR / "ot_input.json"

ALIGNED_H5AD = Path("inputs/chicken_heart_aligned.h5ad")
MODEL_DIR = Path("inputs/chicken_heart_model")
TRAIN_OUTPUT_DIR = Path("tutorial_outputs/chicken_heart/training_run")
DOWNSTREAM_OUTPUT_DIR = Path("tutorial_outputs/chicken_heart/downstream_run")

RUN_PREPARATION = False
REPAIR_LEGACY_D7_LEFT_RIGHT = False
RUN_TRAINING = False
LOAD_MODEL = False
RUN_DOWNSTREAM = False

DEVICE_OVERRIDE: str | None = None
DEVICE = DEVICE_OVERRIDE or ("cuda" if torch.cuda.is_available() else "cpu")
GRAPH_DATABASE = cb.pp.bundled_graph_database_path(DATASET_PRESET)
pd.Series({"device": DEVICE, "preset": DATASET_PRESET})


## 2. Prepare or validate data

`prepare_chicken_heart_input` recovers the reference spot roster and raw counts. The optional legacy D7 repair remains off for current inputs. `prepare_chicken_heart_ot_input` then creates the `spatial_ot_input` used by the packaged alignment step.


In [ ]:
if RUN_PREPARATION:
    cb.pp.prepare_chicken_heart_input(
        raw_dir=RAW_DIR,
        metadata_h5ad=METADATA_H5AD,
        aligned_reference_h5ad=REFERENCE_ALIGNMENT_H5AD,
        output_h5ad=PREPARED_H5AD,
        output_table=PREPARED_TABLE,
        manifest_path=PREPARED_MANIFEST,
        graph_database=GRAPH_DATABASE,
        repair_legacy_d7_left_right=REPAIR_LEGACY_D7_LEFT_RIGHT,
    )
    cb.pp.prepare_chicken_heart_ot_input(
        input_h5ad=PREPARED_H5AD,
        output_h5ad=WORKFLOW_INPUT_H5AD,
        output_table=WORKFLOW_INPUT_TABLE,
        manifest_path=WORKFLOW_INPUT_MANIFEST,
    )

preparation_rows = []
if PREPARED_H5AD.is_file():
    prepared = sc.read_h5ad(PREPARED_H5AD)
    prepared_check = cb.pp.validate_prepared_chicken_heart_input(prepared)
    preparation_rows.append(
        {
            "file": PREPARED_H5AD.name,
            "spots": prepared.n_obs,
            "genes": prepared.n_vars,
            "coordinate_policy": prepared_check["coordinate_policy"],
        }
    )
if WORKFLOW_INPUT_H5AD.is_file():
    workflow_input = sc.read_h5ad(WORKFLOW_INPUT_H5AD)
    workflow_input_check = cb.pp.validate_chicken_heart_ot_input(workflow_input)
    preparation_rows.append(
        {
            "file": WORKFLOW_INPUT_H5AD.name,
            "spots": workflow_input.n_obs,
            "genes": workflow_input.n_vars,
            "coordinate_policy": workflow_input_check["coordinate_policy"],
        }
    )
if preparation_rows:
    display(pd.DataFrame(preparation_rows))
else:
    print("Set the input paths and enable RUN_PREPARATION, or provide an existing workflow input.")


## 3. Load the preset and build a plan

The packaged preset supplies the alignment, graph, training, and downstream settings. The plan below changes with `RUN_TRAINING`.


In [ ]:
workflow_config, workflow_source = load_workflow_config(DATASET_PRESET)
if RUN_TRAINING:
    plan_options = WorkflowOptions(
        input_h5ad=WORKFLOW_INPUT_H5AD,
        output_dir=TRAIN_OUTPUT_DIR,
        device=DEVICE,
        train=True,
        steps=("preprocess", "train"),
    )
else:
    plan_options = WorkflowOptions(
        aligned_h5ad=ALIGNED_H5AD,
        model_dir=MODEL_DIR,
        output_dir=DOWNSTREAM_OUTPUT_DIR,
        device=DEVICE,
        steps=("downstream",),
    )
plan = build_workflow_plan(workflow_config, source=workflow_source, options=plan_options)
print(render_workflow_plan(plan))


## 4. Train a new model or load a checkpoint

Leave `RUN_TRAINING=False` to keep the existing checkpoint in `MODEL_DIR` separate from a new training run. Set `LOAD_MODEL=True` to load the selected checkpoint after the aligned input is available.


In [ ]:
active_aligned_h5ad = ALIGNED_H5AD
active_model_dir = MODEL_DIR
training_result = None
if RUN_TRAINING:
    if not WORKFLOW_INPUT_H5AD.is_file():
        raise FileNotFoundError(f"Workflow input not found: {WORKFLOW_INPUT_H5AD}")
    training_result = run_workflow(workflow_config, options=plan_options)
    active_aligned_h5ad = TRAIN_OUTPUT_DIR / "preprocess" / "chicken_heart_aligned.h5ad"
    active_model_dir = TRAIN_OUTPUT_DIR / "training"

loaded = None
aligned = None
if RUN_TRAINING or LOAD_MODEL:
    if not active_aligned_h5ad.is_file():
        raise FileNotFoundError(f"Aligned input not found: {active_aligned_h5ad}")
    if not active_model_dir.is_dir():
        raise FileNotFoundError(f"Model directory not found: {active_model_dir}")
    aligned = sc.read_h5ad(active_aligned_h5ad)
    dataset_settings = workflow_config["dataset"]
    spatial_key = str(dataset_settings["spatial_key"])
    latent_key = str(dataset_settings["obsm_key"])
    spatial_dim = int(aligned.obsm[spatial_key].shape[1])
    latent_dim = int(aligned.obsm[latent_key].shape[1])
    model_dim = spatial_dim + latent_dim
    loaded = cb.tl.load_dynamical_model_from_dir(
        active_model_dir,
        dim=model_dim,
        device=DEVICE,
    )
    display(
        pd.Series(
            {
                "model_directory": active_model_dir.name,
                "model_dimension": model_dim,
                "weight_stage": loaded.weight_stage,
                "score_stage": loaded.score_stage,
            }
        )
    )
else:
    print("Training and model loading are disabled.")


## 5. Run downstream analysis and save outputs

The downstream step uses the selected aligned input and model directory. Tables, figures, generated slices, and `summary.json` are written under `DOWNSTREAM_OUTPUT_DIR / 'downstream'`.


In [ ]:
downstream_result = None
if RUN_DOWNSTREAM:
    if loaded is None:
        raise RuntimeError("Load a model before running downstream analysis.")
    downstream_options = WorkflowOptions(
        aligned_h5ad=active_aligned_h5ad,
        model_dir=active_model_dir,
        output_dir=DOWNSTREAM_OUTPUT_DIR,
        device=DEVICE,
        steps=("downstream",),
    )
    downstream_result = run_workflow(
        workflow_config,
        options=downstream_options,
    )
else:
    print("Downstream analysis is disabled.")


## Outputs


In [ ]:
summary_path = DOWNSTREAM_OUTPUT_DIR / "downstream" / "summary.json"
output_summary = {
    "workflow_input": WORKFLOW_INPUT_H5AD.name,
    "aligned_input": active_aligned_h5ad.name,
    "model_directory": active_model_dir.name,
    "downstream_summary": summary_path.name,
    "summary_exists": summary_path.is_file(),
}
if summary_path.is_file():
    saved_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    output_summary["analysis_count"] = len(saved_summary.get("analyses", {}))
pd.Series(output_summary)
